# Phase 1 – Data Understanding & Quality Checks
**Sigmoid case study – Bike Sharing Demand**

Run the cells top to bottom (`Runtime → Run all`). Each check has an **Expected** line so you can confirm your output matches.

## Step 1 – Upload the dataset ZIP
Running the next cell opens a file picker in Colab. Choose the bike-sharing ZIP (the one containing `hour.csv`, `day.csv` and `Readme.txt`).
If the CSV files are already in the working folder, the cell skips the upload.

In [ ]:
import os, glob, zipfile

if not (os.path.exists("data/hour.csv") and os.path.exists("data/day.csv")):
    try:
        from google.colab import files          # only exists inside Colab
        uploaded = files.upload()               # pick the ZIP
        zip_name = [f for f in uploaded if f.lower().endswith(".zip")][0]
    except ImportError:                         # running outside Colab
        zip_name = glob.glob("*.zip")[0]
    os.makedirs("data", exist_ok=True)
    with zipfile.ZipFile(zip_name) as z:
        for member in z.namelist():             # flatten any sub-folder inside the zip
            name = os.path.basename(member)
            if name in ("hour.csv", "day.csv", "Readme.txt"):
                with z.open(member) as src, open(os.path.join("data", name), "wb") as dst:
                    dst.write(src.read())

print(sorted(os.listdir("data")))

**Expected:** `['Readme.txt', 'day.csv', 'hour.csv']`

## Step 2 – Read the documentation (don't guess variable meanings)

In [ ]:
readme = open("data/Readme.txt", encoding="latin-1").read()
start = readme.find("Dataset characteristics"); end = readme.find("License")
print(readme[start:end])

Key facts from it: `cnt = casual + registered`; `temp`, `atemp`, `hum`, `windspeed` are normalised (divided by 41, 50, 100, 67); `weathersit` has 4 levels.

## Step 3 – Load data, shape and data types

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)

hour = pd.read_csv("data/hour.csv", parse_dates=["dteday"])
day  = pd.read_csv("data/day.csv",  parse_dates=["dteday"])

print("hour.csv:", hour.shape, "| day.csv:", day.shape)
display(hour.head())
print(hour.dtypes)

**Expected:** `hour.csv: (17379, 17) | day.csv: (731, 16)`. All columns numeric except `dteday` (date).

## Step 4 – Missing values and duplicates

In [ ]:
for name, df in [("hour", hour), ("day", day)]:
    print(f"{name}: nulls = {df.isna().sum().sum()}, "
          f"duplicate rows = {df.duplicated().sum()}, "
          f"duplicates ignoring 'instant' = {df.drop(columns='instant').duplicated().sum()}")

**Expected:** all zeros for both files.

## Step 5 – Consistency checks (leakage + hourly vs daily)

In [ ]:
print("hour: cnt == casual + registered on every row ->", (hour.cnt == hour.casual + hour.registered).all())
print("day : cnt == casual + registered on every row ->", (day.cnt == day.casual + day.registered).all())

hourly_sum = hour.groupby("dteday").cnt.sum()
print("hourly totals equal daily file for all days  ->", (hourly_sum.values == day.set_index("dteday").cnt.values).all())

**Expected:** `True`, `True`, `True`.
**Why it matters:** because `casual + registered` equals the target exactly, using them as features would be **target leakage** → they are excluded from the model.

## Step 6 – Valid ranges for every column

In [ ]:
checks = {
    "season in 1-4":      hour.season.between(1, 4).all(),
    "yr in 0-1":          hour.yr.isin([0, 1]).all(),
    "mnth in 1-12":       hour.mnth.between(1, 12).all(),
    "hr in 0-23":         hour.hr.between(0, 23).all(),
    "weekday in 0-6":     hour.weekday.between(0, 6).all(),
    "holiday/workingday binary": hour.holiday.isin([0,1]).all() and hour.workingday.isin([0,1]).all(),
    "weathersit in 1-4":  hour.weathersit.between(1, 4).all(),
    "temp/atemp/hum/wind in [0,1]": hour[["temp","atemp","hum","windspeed"]].apply(lambda c: c.between(0,1).all()).all(),
    "counts non-negative": (hour[["casual","registered","cnt"]] >= 0).all().all(),
}
for k, v in checks.items():
    print(f"{k:32s} {v}")

display(hour.drop(columns=["instant", "dteday"]).describe().T[["min", "max", "mean"]].round(3))

**Expected:** all `True`. Ranges are fine – but *in-range* does not mean *plausible*, see Step 8.

## Step 7 – Date coverage and missing hours

In [ ]:
print("Date range:", hour.dteday.min().date(), "to", hour.dteday.max().date(), "| days:", hour.dteday.nunique())
print("Expected hours:", hour.dteday.nunique() * 24, "| present:", len(hour),
      "| missing:", hour.dteday.nunique() * 24 - len(hour))

hours_per_day = hour.groupby("dteday").size()
print("\nDays by number of recorded hours:\n", hours_per_day.value_counts().sort_index())
print("\nDays with fewer than 22 hours:\n", hours_per_day[hours_per_day < 22])

# Which clock hours are missing on ordinary days?
full = pd.MultiIndex.from_product([hours_per_day.index, range(24)], names=["dteday", "hr"])
missing = full.difference(pd.MultiIndex.from_frame(hour[["dteday", "hr"]])).to_frame(index=False)
storm_days = hours_per_day[hours_per_day < 22].index
print("\nMissing hours by clock hour (storm days excluded):")
print(missing[~missing.dteday.isin(storm_days)].hr.value_counts().sort_index().to_dict())
print("\nSmallest cnt in the file:", hour.cnt.min())

**Expected:** 2011-01-01 → 2012-12-31, 731 days, **165 missing hours**.
Short days: 2011-01-18, 2011-01-26/27 (snowstorm), 2011-02-22, 2011-08-27/28 (Hurricane Irene), 2012-10-29/30 (Hurricane Sandy – only 1 and 11 hours).
On normal days the missing hours are mostly **02:00–05:00**, and the smallest `cnt` is 1 → hours with **zero rentals were probably never recorded**.

## Step 8 – Implausible values (in range but physically wrong)

In [ ]:
print("Rows with humidity = 0:", (hour.hum == 0).sum(), "on dates", hour.loc[hour.hum == 0, "dteday"].dt.date.unique())
print("Rows with windspeed = 0:", (hour.windspeed == 0).sum(),
      f"({(hour.windspeed == 0).mean():.1%})")
print("Smallest windspeed values:", sorted(hour.windspeed.unique())[:4])
print("\nweathersit counts:\n", hour.weathersit.value_counts().sort_index())

**Expected:**
- `hum = 0` on **22 rows, all on 2011-03-10** → impossible (0% humidity) → sensor error → impute later, don't delete the day.
- `windspeed = 0` on **2,180 rows (12.5%)**, and the next value is 0.0896 → a gap that suggests "not recorded/calm". Flag it.
- `weathersit = 4` appears only **3 times** → merge into 3 for modelling.

## Step 9 – Confirm the weekday coding

In [ ]:
first = hour.iloc[0]
print(first.dteday.date(), "is a", first.dteday.day_name(), "-> weekday code", first.weekday)

# workingday should be 1 exactly when it is Mon-Fri and not a holiday
derived = ((hour.weekday.between(1, 5)) & (hour.holiday == 0)).astype(int)
print("workingday matches (Mon-Fri & not holiday):", (derived == hour.workingday).all())

**Expected:** 2011-01-01 is a Saturday → code 6, so **0 = Sunday … 6 = Saturday**; workingday check `True`.

## Step 10 – Target distribution (outliers?)

In [ ]:
print(hour.cnt.describe().round(1))
print("Skewness:", round(hour.cnt.skew(), 2))
print("Quantiles:", hour.cnt.quantile([0.5, 0.9, 0.99]).round(0).to_dict())

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(hour.cnt, bins=50); ax[0].set_title("Hourly rentals (cnt)"); ax[0].set_xlabel("rentals per hour")
ax[1].boxplot(hour.cnt, vert=False); ax[1].set_title("Boxplot of cnt")
plt.tight_layout(); plt.show()

# Where do the 'outliers' happen?
q1, q3 = hour.cnt.quantile([0.25, 0.75]); upper = q3 + 1.5 * (q3 - q1)
out = hour[hour.cnt > upper]
print(f"IQR outliers (> {upper:.0f}): {len(out)} rows")
print("Share on working days:", round(out.workingday.mean(), 2))
print("Most common hours:", out.hr.value_counts().head(4).to_dict())

**Expected:** median 142, p99 ≈ 782, max 977, skew ≈ 1.28.
505 IQR "outliers" (> 642/hr): **82% on working days**, concentrated at **5 PM, 6 PM and 8 AM** → genuine commute peaks, **not errors → keep them**.

## Step 11 – Year-on-year growth (affects how we split train/test)

In [ ]:
print(hour.groupby("yr").cnt.mean().rename({0: "2011", 1: "2012"}).round(1))
growth = hour[hour.yr == 1].cnt.mean() / hour[hour.yr == 0].cnt.mean() - 1
print(f"Growth: {growth:+.0%}")

day.set_index("dteday").cnt.rolling(7).mean().plot(figsize=(12, 3), title="Daily rentals, 7-day rolling mean")
plt.ylabel("rentals per day"); plt.show()

**Expected:** 143.8 → 234.7 per hour (**+63%**). Training only on 2011 and testing on 2012 would force the model to extrapolate a level it has never seen.

## Step 12 – The key early signal: working vs non-working hourly pattern

In [ ]:
profile = hour.groupby(["hr", "workingday"]).cnt.mean().unstack()
profile.columns = ["Non-working day", "Working day"]
print(profile.loc[[3, 8, 12, 17, 18]].round(0))

profile.plot(figsize=(10, 4), marker="o", title="Average rentals by hour")
plt.xlabel("hour of day"); plt.ylabel("avg rentals"); plt.xticks(range(24)); plt.grid(alpha=.3); plt.show()

**Expected:** working days 8 AM ≈ 477 and 5 PM ≈ 525 (sharp commute spikes); non-working days peak ≈ 366 around midday.

## Step 13 – Quick sanity check on daily totals (formal test comes in Phase 2)

In [ ]:
from scipy import stats
wd, nwd = day.cnt[day.workingday == 1], day.cnt[day.workingday == 0]
print(f"Working days: n={len(wd)}, mean={wd.mean():.0f} | Non-working: n={len(nwd)}, mean={nwd.mean():.0f}")
print(stats.ttest_ind(wd, nwd, equal_var=False))

**Expected:** means ≈ 4,585 vs 4,330, Welch t ≈ 1.60, **p ≈ 0.11** → daily *volume* is not clearly different, but the hourly *timing* (Step 12) is very different.
That is why the Phase 2 hypothesis focuses on **peak concentration**, which speaks directly to "no bike when I need one, idle bikes elsewhere".

## Phase 1 summary – what you have verified
| Check | Result |
|---|---|
| Nulls / duplicates | none |
| Leakage | `casual + registered = cnt` → excluded; `instant` excluded |
| Missing hours | 165 (mostly zero-demand night hours + storm shutdowns) |
| Bad values | `hum=0` on 2011-03-10 (impute), `windspeed=0` 12.5% (flag), `weathersit=4` ×3 (merge) |
| Outliers | genuine commute peaks → kept |
| Trend | +63% growth 2011→2012 → affects train/test split |
| Key insight | similar daily volume, very different hourly timing |